# 05 — Data audit summary
Kết quả khảo sát dữ liệu hiện tại được ghi ngay dưới đây. Chạy các ô code để tính lại báo cáo Markdown và bảng trực tiếp trong notebook khi dữ liệu thay đổi. Hai dataset được báo cáo tách biệt, không trộn nhãn hay thay đổi split.

## DeepPCB
- **1.500 ảnh, 1.500 label, 10.013 box** (train 1.200 ảnh/8.024 box; valid 150/1.005; test 150/984).
- Class nhiều nhất: **mousebite** (1.965 box); ít nhất: **copper** (1.474 box).
- Annotation: **No annotation issues found** theo các kiểm tra trong notebook 02.
- Size: small 3.204; **medium 6.802 (67,9%)**; large 7 box.

## PKU
- **3.504 ảnh, 3.504 label, 14.426 box** (train 2.772 ảnh/11.812 box; valid 366/1.307; test 366/1.307).
- Class nhiều nhất: **Spurious_copper** (2.462 box); ít nhất: **Open_circuit** (2.338 box).
- Annotation: **No annotation issues found** theo các kiểm tra trong notebook 02.
- Size: small 202; **medium 13.376 (92,7%)**; large 848 box.

## Nhận định
- **PKU có thể là dataset chính**: nhiều ảnh và box hơn, class khá cân bằng; cần kiểm tra ảnh tăng cường có liên quan giữa các split trước khi dùng số liệu đánh giá để kết luận chất lượng model.
- **DeepPCB phù hợp làm dataset phụ/external test** để kiểm tra khả năng tổng quát hóa. Không mặc định đồng nhất các nhãn gần nghĩa như `pin-hole` và `Missing_hole`; cần xác nhận ánh xạ class khi dùng chung ở bước sau.

*Số lượng box ở đây là số dòng annotation không rỗng; kết quả size tính từ box hợp lệ. Chưa chuyển đổi annotation, chia lại tập hay huấn luyện model.*

In [1]:
import json
from pathlib import Path
notebook_path = next((p / 'scripts' / '01_inventory.ipynb' for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / '01_inventory.ipynb').is_file()), None)
if notebook_path is None:
    raise FileNotFoundError('01_inventory.ipynb not found: run inside the project.')
exec(compile(''.join(json.loads(notebook_path.read_text(encoding='utf-8'))['cells'][1]['source']), str(notebook_path), 'exec'))


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
audits = all_audits()
for name, result in audits.items():
    counts = result['counts'][['images', 'labels', 'boxes']].sum()
    frequencies = result['by_class'].groupby('class', sort=False)['boxes'].sum()
    most = ', '.join(frequencies[frequencies == frequencies.max()].index)
    least = ', '.join(frequencies[frequencies == frequencies.min()].index)
    issues = result['issues']
    issue_text = ('No annotation issues found' if issues.empty else '; '.join(f'{kind}: {count}' for kind, count in issues['issue'].value_counts().items()))
    area = result['boxes']['area_px']
    bins = pd.Series({'small': int((area < 1024).sum()), 'medium': int(((area >= 1024) & (area <= 9216)).sum()), 'large': int((area > 9216).sum())})
    largest_bin = ', '.join(bins[bins == bins.max()].index) if len(area) else 'không có box hợp lệ'
    pct = 100 * bins.max() / len(area) if len(area) else 0
    display(Markdown(f'## {name}\n'
                     f'- **Quy mô:** {counts["images"]:,} ảnh; {counts["labels"]:,} label; {counts["boxes"]:,} dòng bounding box, trong đó {len(area):,} box hợp lệ cho phân tích kích thước.\n'
                     f'- **Class nhiều nhất:** {most} ({frequencies.max():,} box); **ít nhất:** {least} ({frequencies.min():,} box).\n'
                     f'- **Annotation:** {issue_text}.\n'
                     f'- **Size bin:** small {bins["small"]:,}, medium {bins["medium"]:,}, large {bins["large"]:,}; nhiều nhất là **{largest_bin}** ({pct:.1f}% box hợp lệ).'))
    display(Markdown('### Theo split'))
    display(result['counts'].set_index('split')[['images', 'labels', 'boxes']])
    display(Markdown('### Theo class và size'))
    display(pd.DataFrame({'boxes_per_class': frequencies}))
    display(bins.rename('valid_boxes').to_frame())


## DeepPCB
- **Quy mô:** 1,500 ảnh; 1,500 label; 10,013 dòng bounding box, trong đó 10,013 box hợp lệ cho phân tích kích thước.
- **Class nhiều nhất:** mousebite (1,965 box); **ít nhất:** copper (1,474 box).
- **Annotation:** No annotation issues found.
- **Size bin:** small 3,204, medium 6,802, large 7; nhiều nhất là **medium** (67.9% box hợp lệ).

### Theo split

,images,labels,boxes
split,,,
train,1200,1200,8024
valid,150,150,1005
test,150,150,984


### Theo class và size

,boxes_per_class
class,
open,1942
short,1506
mousebite,1965
spur,1625
copper,1474
pin-hole,1501


,valid_boxes
small,3204
medium,6802
large,7


... (2 outputs left out)

,images,labels,boxes
split,,,
train,2772,2772,11812
valid,366,366,1307
test,366,366,1307


### Theo class và size

,boxes_per_class
class,
Missing_hole,2434
Mouse_bite,2410
Open_circuit,2338
Short,2396
Spur,2386
Spurious_copper,2462


,valid_boxes
small,202
medium,13376
large,848


In [3]:
pku, deep = audits['PKU'], audits['DeepPCB']
pku_images = int(pku['counts']['images'].sum())
deep_images = int(deep['counts']['images'].sum())
pku_issues, deep_issues = len(pku['issues']), len(deep['issues'])
recommendation = ('Có thể ưu tiên PKU làm dataset chính để thử nghiệm nhờ quy mô lớn hơn DeepPCB và ba split sẵn có' if pku_images > deep_images else 'Chưa đủ cơ sở ưu tiên PKU theo quy mô ảnh; cần xem thêm chất lượng và mục tiêu triển khai')
display(Markdown(f'## Đề xuất sử dụng dữ liệu\n'
                 f'- **PKU làm dataset chính?** {recommendation} ({pku_images:,} so với {deep_images:,} ảnh). Cần xem xét {pku_issues:,} lỗi được báo cáo trước khi đưa vào training, và đánh giá mức phù hợp với dữ liệu thực tế. Bản tăng cường dữ liệu có thể chứa ảnh liên quan nhau: nên kiểm tra rò rỉ giữa các split trước khi diễn giải độ chính xác.\n'
                 f'- **DeepPCB làm dataset phụ/external test?** Có thể dùng làm bộ đánh giá ngoài hoặc bộ phụ để kiểm tra khả năng tổng quát hóa (hiện báo cáo {deep_issues:,} lỗi). Cần ánh xạ tên class theo ngữ nghĩa sau audit (ví dụ `pin-hole` không mặc nhiên tương đương `Missing_hole`) và giữ test độc lập; không gộp nhãn hoặc train ở giai đoạn này.'))


## Đề xuất sử dụng dữ liệu
- **PKU làm dataset chính?** Có thể ưu tiên PKU làm dataset chính để thử nghiệm nhờ quy mô lớn hơn DeepPCB và ba split sẵn có (3,504 so với 1,500 ảnh). Cần xem xét 0 lỗi được báo cáo trước khi đưa vào training, và đánh giá mức phù hợp với dữ liệu thực tế. Bản tăng cường dữ liệu có thể chứa ảnh liên quan nhau: nên kiểm tra rò rỉ giữa các split trước khi diễn giải độ chính xác.
- **DeepPCB làm dataset phụ/external test?** Có thể dùng làm bộ đánh giá ngoài hoặc bộ phụ để kiểm tra khả năng tổng quát hóa (hiện báo cáo 0 lỗi). Cần ánh xạ tên class theo ngữ nghĩa sau audit (ví dụ `pin-hole` không mặc nhiên tương đương `Missing_hole`) và giữ test độc lập; không gộp nhãn hoặc train ở giai đoạn này.